# 2. Residual streams and the logit lens

**Question:** what vocabulary tokens become likely if we decode an intermediate residual state using the model's final normalization and unembedding? This is the logit lens baseline.

Run the setup cell from notebook 1 first, then open this notebook in the same runtime. If running it alone, repeat the setup cell below.

In [ ]:
import sys
from pathlib import Path
repo = Path('/content/slm-jspace-viewer')
if not repo.exists():
    !git clone --branch feature/colab-jspace-tutorial --single-branch https://github.com/prithvimk/slm-jspace-viewer.git {repo}
%cd /content/slm-jspace-viewer
!git fetch origin feature/colab-jspace-tutorial
!git checkout feature/colab-jspace-tutorial
!{sys.executable} -m pip -q install uv
!{sys.executable} -m uv export --locked --no-hashes --no-dev --group tutorial -o /tmp/jspace-colab.txt
!{sys.executable} -m pip -q install -r /tmp/jspace-colab.txt
!{sys.executable} -m pip -q install -e .

In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import Markdown, display
from slm_jspace.lenses import LogitLens, top_k
from tutorials.lib.models import load_tutorial_model, residual_trace
from tutorials.lib.visuals import residual_projection

tutorial_model = load_tutorial_model()
prompt = 'The capital of France is'
token_ids, residuals = residual_trace(tutorial_model, prompt)
tokens = [tutorial_model.tokenizer.decode([token]) for token in token_ids]
layers = list(range(residuals.shape[0]))
centered = residuals.reshape(-1, residuals.shape[-1]).numpy()
centered = centered - centered.mean(axis=0, keepdims=True)
_, _, right = np.linalg.svd(centered, full_matrices=False)
projection = (centered @ right[:2].T).reshape(residuals.shape[0], residuals.shape[1], 2)
display(Markdown('**Question:** how does this prompt move through residual space as depth increases?'))
display(residual_projection(projection, layers, tokens))

In [ ]:
layer = widgets.IntSlider(min=0, max=len(layers)-1, value=len(layers)//2, description='Layer:')
position = widgets.IntSlider(min=0, max=len(tokens)-1, value=len(tokens)-1, description='Position:')
output = widgets.Output()

def inspect(_: object = None):
    with output:
        output.clear_output(wait=True)
        vector = residuals[layer.value:layer.value+1, position.value:position.value+1].to(next(tutorial_model.model.parameters()).device)
        scores = LogitLens(tutorial_model.model).read(vector, [layer.value])[0, 0]
        display(Markdown(f'**Question:** what does the logit lens decode at layer {layer.value}, token `{tokens[position.value]!r}`?'))
        for item in top_k(scores, tutorial_model.tokenizer, count=12):
            print(f"{item['token']!r:18} score={item['score']:.3f}")

layer.observe(inspect, names='value'); position.observe(inspect, names='value')
display(widgets.VBox([widgets.HBox([layer, position]), output]))
inspect()

## Takeaway

The logit lens is intentionally simple. It assumes intermediate residuals are already expressed in final-layer coordinates. The Jacobian lens in notebook 3 tests a transport-aware alternative.